# 13.1 - Manual State-Machine Agent

**Phase:** 13 - LangGraph / Stateful Workflows

**Status:** VERIFIED

---

## 1. What Are We Solving?

A workflow that must branch and loop cannot be written as one long straight-line script. A state machine models it as *states* and *transitions* driven by a tiny main loop. We hand-roll the machine first so LangGraph's abstraction in unit 13.2 feels earned, not mysterious.

## 2. Why Does This Matter?

## 3. Prerequisites

Phase 12 (LLM application patterns), basic Python functions and dictionaries.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Define workflow state as a dictionary of typed fields
- Write node functions that receive state and return new state
- Write a router that decides which node runs next
- Drive the loop to a terminal state, guarding against infinite loops
- Read a state trace to debug routing bugs

## 5. Mental Model

A state machine is a vending machine: it has states (idle, processing, waiting, done), each state knows which transitions are valid, input triggers a transition, and it never skips a state.

```
state = { ... fields ... }
while True:
    next_node = router(state)          # which step runs next?
    if next_node == '__end__': break   # terminal state
    state = NODES[next_node](state)    # run that step, update state
```


## 6. Environment + LLM Helper

All generation calls go through the Groq free tier (`GROQ_API_KEY` in `.env`). Without a key the helper returns a deterministic mock, so the notebook still runs offline.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: LangGraph is a stateful orchestration framework."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:  # network / quota / model errors -> never crash the notebook
        return f"[llm-error: {type(e).__name__}]"

print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


## 7. Build the Machine: Nodes, Router, Main Loop

A question-answering agent: retrieve documents, then generate an answer, then stop.

In [2]:
state = {
    "query": "What is LangGraph?",
    "scratchpad": [],
    "answer": None,
    "step": 0,
    "max_steps": 3,
}


def retrieve(state: dict) -> dict:
    state["scratchpad"].append(f"[{state['step']}] retrieved 2 docs")
    state["step"] += 1
    return state


def generate(state: dict) -> dict:
    state["answer"] = llm(f"Answer concisely: {state['query']}")
    state["scratchpad"].append(f"[{state['step']}] generated answer")
    state["step"] += 1
    return state


def router(state: dict) -> str:
    if state["answer"] is not None:
        return "__end__"
    if state["step"] >= state["max_steps"]:
        return "__end__"
    if not state["scratchpad"]:
        return "retrieve"
    return "generate"


NODES = {"retrieve": retrieve, "generate": generate}

while True:
    choice = router(state)
    if choice == "__end__":
        break
    state = NODES[choice](state)

print("ANSWER:", state["answer"])
print("TRACE:")
for line in state["scratchpad"]:
    print("   ", line)
print("total steps:", state["step"])


ANSWER: **LangGraph** is a framework for building and orchestrating language‑model applications as directed graphs. It lets developers define nodes (e.g., prompts, LLM calls, logic, I/O) and edges (control flow, data flow), then execute the graph to create complex, stateful, multi‑step LLM workflows.
TRACE:
    [0] retrieved 2 docs
    [1] generated answer
total steps: 2


## 8. Real-World Example: Ticket Triage Agent

Classify an incoming ticket, route it to the right department handler, and stop once answered.

In [3]:
def classify(state: dict) -> dict:
    q = state["query"].lower()
    if any(w in q for w in ["refund", "billing", "charge"]):
        state["category"] = "billing"
    elif any(w in q for w in ["error", "bug", "crash"]):
        state["category"] = "tech"
    else:
        state["category"] = "faq"
    return state


def handle_billing(state):
    state["answer"] = "BILLING> " + llm(f"Reply as billing support: {state['query']}")
    return state


def handle_tech(state):
    state["answer"] = "TECH> " + llm(f"Reply as tech support: {state['query']}")
    return state


def handle_faq(state):
    state["answer"] = "FAQ> " + llm(f"Reply from the FAQ: {state['query']}")
    return state


def triage_router(state):
    if state.get("answer"):
        return "__end__"
    if state.get("category") == "billing":
        return "billing"
    if state.get("category") == "tech":
        return "tech"
    return "faq"


state = {"query": "My card was charged twice, can I get a refund?", "category": None, "answer": None}
TRIAGE = {"billing": handle_billing, "tech": handle_tech, "faq": handle_faq}

while True:
    step = triage_router(state)
    if step == "__end__":
        break
    state = TRIAGE[step](state)

print("Category:", state["category"])
print("Answer:", state["answer"])

Category: None
Answer: FAQ> **FAQ – Duplicate Charge on Your Card**

**Q: My card was charged twice, can I get a refund?**  
**A:** Yes – we’ll help you get a refund for any duplicate charge. Follow the steps below to resolve the issue quickly.

---

### 1. Verify the Charge
| What to check | Why it matters |
|---------------|----------------|
| **Transaction date & time** | Duplicate charges often appear as a “pending” transaction that later settles. |
| **Merchant name** | Confirms it’s the same merchant. |
| **Amount** | Ensure the amounts match exactly. |
| **Status** | Pending, settled, or reversed. |

> **Tip:** If the duplicate is still *pending*, it may automatically cancel after 7–10 days. If it’s already settled, you’ll need to request a refund.

---

### 2. Contact the Merchant First (If Possible)
- **Why?** Many merchants can reverse the duplicate charge immediately.
- **How?** Use the merchant’s customer support number or email (usually found on the receipt or their websit


## Common Mistakes

- **Return `None` from a node** — the graph silently drops the update. Always `return state`.
- **Mutating state in the router** — routers must be side-effect free.
- **Forgetting the terminal condition** — cycles run forever without an iteration guard.
- **Typo in a state key** — `TypedDict` catches it at compile time; plain dicts do not.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| `KeyError` on state | Field name mismatch | Match state keys to the `TypedDict` exactly |
| Graph won't compile | Node/referenced name typo | Check every string passed to `add_node`/`add_edge` |
| Node output ignored | Node returns `None` or a partial dict | Always `return state` (or a merge-able partial) |
| Infinite loop | No convergence guard | Add `max_steps` to state and check it in the router |
| Wrong branch taken | Router priority bug | Unit-test the router on every input variant |

## Best Practices

- Define all state fields upfront with defaults in a `TypedDict`.
- Keep node functions pure and focused: one responsibility each.
- Name nodes descriptively (`retrieve`, `generate`, not `step1`).
- Always add an iteration guard on loops.
- Inspect the graph with `app.get_graph().draw_mermaid()`.

## Hands-On Practice

1. **Basic:** Rerun the examples with new inputs; verify the trace.
2. **Guided:** Add a node that validates output before terminating.
3. **Independent:** Build a 3-step pipeline (fetch -> process -> summarize) with a retry node.
4. **Realistic:** Turn the example into a multi-department support agent.
5. **Challenge:** Save/load the state dict to JSON and resume the workflow from a checkpoint.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
